# Introduction to Machine Learning · MDMC 2026
## Day 3 · Hands-on 3: Decoding and Dimension

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/myscience/intro-ml-mdmc-2026/blob/main/notebooks/N3_decoding_and_dimension.ipynb)

You will read the direction of a drifting grating from the activity of 11,302 neurons in mouse visual cortex, and evaluate the decoder honestly with Tuesday's toolkit: a pipeline, cross-validation, a baseline and the confusion matrix (slides *Reading Out the Stimulus* and *Hands-On 3*).
If you have time, you will pit this morning's models against each other, and go back to the geometry of the data: the loop of the 32 directions (*A Loop in Neural Space*) and its intrinsic dimension (*Two Neighbours Are Enough*, *Noise and Scale*).

**Data.** Stringer, Pachitariu, Steinmetz, Carandini & Harris (2019), *High-dimensional geometry of population responses in visual cortex*, Nature 571, 361–365. Session `ori32_M160825_MP027_2016-12-15`, from figshare, [doi:10.25378/janelia.6845348.v4](https://doi.org/10.25378/janelia.6845348.v4). Licence [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/): credit the authors in anything you make from it.

| Tier | What | Time |
|---|---|---|
| ★ Core | Everyone. Fill in the blanks marked `____` or `# YOUR CODE HERE`. | about 35 min |
| ★★ Stretch | Fewer hints, small design decisions. | about 20 min |
| ★★★ Challenge | Open-ended, for when you are done. | as long as you like |

Every exercise ends with a **checkpoint**: an `assert` that passes, or a "you should see …". Run the cells from top to bottom.

## Setup
Run this cell once.

In [ ]:
import hashlib
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import scipy.io
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from sklearn.datasets import make_swiss_roll
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 0
rng = np.random.default_rng(SEED)

# the course palette (light-theme values)
YELLOW, CYAN, MAGENTA, LIME = "#D99E00", "#0094AE", "#D1006C", "#7E8E00"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 11})

## The data

A mouse looks at drifting gratings, stripes moving in one of 32 directions (0°, 11.25°, …, 348.75°), while a two-photon microscope records 11,302 neurons in its primary visual cortex. Each direction is shown 64 times, in random order; every so often the screen is simply gray. For each trial and each neuron we get one number: how active the neuron was during the stimulus.

The next cell downloads the file (117 MB, once), checks it against the checksum published on figshare, and loads it. A public file with a DOI, a licence and a checksum: that is FAIR data in action.

In [ ]:
URL = "https://ndownloader.figshare.com/files/12462734"
MD5 = "4fe52f03c4f53adf15b9c7c49f6b2b5d"
path = os.environ.get("ORI32_PATH", "ori32_M160825_MP027_2016-12-15.mat")
if not os.path.exists(path):
    print("Downloading 117 MB from figshare (only once) ...")
    urllib.request.urlretrieve(URL, path)
with open(path, "rb") as f:
    assert hashlib.md5(f.read()).hexdigest() == MD5, f"Checksum mismatch: delete {path} and run this cell again."

stim = scipy.io.loadmat(path, squeeze_me=True, struct_as_record=False)["stim"]
resp_all = stim.resp                  # trials x neurons
istim = stim.istim.astype(int)        # stimulus id of each trial: 1-32 = the directions, 33 = gray screen
directions = np.arange(32) * 11.25    # direction of class c, in degrees

print("responses:", resp_all.shape, "(trials x neurons)")
print("trials per stimulus id:", np.bincount(istim)[1:])

Plotting helpers: run this cell, no need to read it.

In [ ]:
def deg_ticks(ax, axis="x"):
    """Label class indices 0, 8, 16, 24 as 0, 90, 180, 270 degrees."""
    getattr(ax, f"set_{axis}ticks")([0, 8, 16, 24])
    getattr(ax, f"set_{axis}ticklabels")(["0°", "90°", "180°", "270°"])


def plot_tuning(tuning, X):
    """Tuning curves of the three neurons whose mean response changes most with direction."""
    best = np.argsort(tuning.std(axis=0) / X.std(axis=0))[-3:][::-1]
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for n, col in zip(best, [YELLOW, CYAN, MAGENTA]):
        ax.plot(directions, tuning[:, n], "o-", ms=4, color=col, label=f"neuron {n}")
    ax.set(xlabel="grating direction", ylabel="mean response", xticks=[0, 90, 180, 270])
    ax.set_xticklabels(["0°", "90°", "180°", "270°"])
    ax.legend(frameon=False)
    plt.show()


def plot_std(std):
    fig, ax = plt.subplots(figsize=(7, 2.8))
    ax.hist(std, bins=np.logspace(-1.5, 3, 60), color=CYAN)
    ax.set(xscale="log", xlabel="standard deviation of a neuron's responses (log scale)", ylabel="neurons")
    plt.show()


def plot_confusion(conf):
    fig, ax = plt.subplots(figsize=(5.6, 4.8))
    im = ax.imshow(conf, cmap="magma_r", norm=LogNorm(vmin=1e-2, vmax=1))
    deg_ticks(ax, "x"); deg_ticks(ax, "y")
    ax.set(xlabel="predicted direction", ylabel="true direction")
    fig.colorbar(im, label="fraction of the row (log scale)")
    plt.show()


def plot_steps(step):
    frac = np.bincount(step, minlength=17) / len(step)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(np.arange(17), 100 * frac, color=[CYAN if s else LIME for s in range(17)])
    ax.set(yscale="log", ylabel="% of trials", xlabel="error, in steps of 11.25° (0 = correct, 16 = opposite)",
           xticks=range(17))
    plt.show()


def plot_models(results, chance):
    names = list(results)
    m = [100 * results[n].mean() for n in names]
    s = [100 * results[n].std() for n in names]
    fig, ax = plt.subplots(figsize=(7, 0.6 * len(names) + 1))
    ax.barh(names, m, xerr=s, color=[YELLOW, CYAN, MAGENTA, LIME][: len(names)])
    ax.axvline(100 * chance, color="k", ls="--", lw=1)
    ax.text(100 * chance + 1, len(names) - 0.6, "chance", fontsize=9)
    for i, (v, e) in enumerate(zip(m, s)):
        ax.text(v + e + 1.5, i, f"{v:.1f}%", va="center")
    ax.set(xlabel="5-fold accuracy (%)", xlim=(0, 100))
    ax.invert_yaxis()
    plt.show()


def plot_session(correct_shuffled, correct_blocked, w=100):
    """Accuracy along the session, as a moving average over w trials."""
    kern = np.ones(w) / w
    t = np.arange(w // 2, len(correct_shuffled) - w // 2 + 1)
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.plot(t, 100 * np.convolve(correct_shuffled, kern, "valid"), color=CYAN, label="shuffled folds")
    ax.plot(t, 100 * np.convolve(correct_blocked, kern, "valid"), color=MAGENTA, label="blocked folds")
    for b in np.cumsum([len(i) for i in np.array_split(np.arange(len(correct_shuffled)), 5)])[:-1]:
        ax.axvline(b, color="0.7", lw=1, ls=":")
    ax.set(xlabel="trial (in the order shown to the mouse)", ylabel=f"accuracy (%), {w}-trial average", ylim=(40, 100))
    ax.legend(frameon=False, loc="lower right")
    plt.show()


def plot_loop(P3):
    """The 32 direction averages on PC1-PC3, coloured by direction and joined in order."""
    fig = plt.figure(figsize=(6.5, 5.5))
    ax = fig.add_subplot(projection="3d")
    xyz = P3[:, [0, 2, 1]]
    seq = list(range(32)) + [0]
    ax.plot(*xyz[seq].T, color="0.6", lw=1.2)
    ring = LinearSegmentedColormap.from_list("ring", [YELLOW, MAGENTA, CYAN, LIME, YELLOW])    # cyclic, as on the slide
    sc = ax.scatter(*xyz.T, c=directions, cmap=ring, vmin=0, vmax=360, s=60, depthshade=False)
    for c in (0, 8):
        ax.text(*xyz[c], f"  {directions[c]:.0f}°", fontsize=10)
    ax.set(xlabel="PC1", ylabel="PC3", zlabel="PC2")
    ax.view_init(elev=30, azim=-120)
    fig.colorbar(sc, shrink=0.6, label="direction (°)", ticks=[0, 90, 180, 270, 360])
    plt.show()


def plot_twonn(x, z, d):
    fig, ax = plt.subplots(figsize=(4.5, 3.5))
    ax.plot(x, z, ".", ms=3, color="0.4")
    ax.plot([0, x.max()], [0, d * x.max()], color=MAGENTA, lw=2, label=f"slope d = {d:.2f}")
    ax.set(xlabel=r"$\log \mu$", ylabel=r"$-\log(1 - F(\mu))$")
    ax.legend(frameon=False)
    plt.show()


def plot_id(KPC, id_single, id_avg16):
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    ax.plot(KPC, id_single, "o-", color="k", label="single trials")
    ax.plot(KPC[: len(id_avg16)], id_avg16, "o-", color=CYAN, label="averages of 16 repeats")
    ax.set(xscale="log", xlabel="principal components kept", ylabel="2NN dimension")
    ax.set_xticks(KPC); ax.set_xticklabels(map(str, KPC)); ax.minorticks_off()
    ax.legend(frameon=False)
    plt.show()

---
# ★ Core: decode the grating direction (about 35 min)

The question of the slide *Reading Out the Stimulus*: from the activity of the neurons on one trial, can a classifier tell which of the 32 directions was on the screen?

### Exercise 1 · Build `X` and `y`

The gray-screen trials (stimulus id 33) have no direction, so we drop them. What is left is a classification problem with 32 classes: `X` holds one row per trial and one column per neuron, and `y` holds the class of each trial, `c = 0, 1, …, 31`, for the direction `c × 11.25°`.

Fill in:
- `keep`: a boolean mask, `True` for the trials where a grating was shown;
- `y`: turn the stimulus ids 1–32 into classes 0–31;
- the `axis` to average over, so that `tuning[c]` is the mean response of every neuron to direction `c` (average over the trials, which are the rows).

In [ ]:
keep = ____
X = resp_all[keep].astype(float)    # trials x neurons
y = istim[keep] - ____
tuning = np.stack([X[y == c].mean(axis=____) for c in range(32)])    # 32 directions x neurons

print("X:", X.shape, " y:", y.shape, " tuning:", tuning.shape)

**Checkpoint.**

In [ ]:
assert X.shape == (2048, 11302), f"X should be 2,048 trials x 11,302 neurons, not {X.shape}"
assert y.min() == 0 and y.max() == 31, "the classes should run from 0 to 31"
assert np.all(np.bincount(y) == 64), "every direction should have 64 trials"
assert tuning.shape == (32, 11302), "tuning should be 32 directions x 11,302 neurons: check the axis"
print("All good: 2,048 trials, 32 directions, 64 repeats each.")

Here are the tuning curves of the three neurons whose response depends most on the direction (slide *Meet the Data*):

In [ ]:
plot_tuning(tuning, X)

**Question.** (a) Each trial is a point in how many dimensions? A logistic regression on the raw neurons would need one weight per neuron and per class: how does that number compare with the 2,048 trials? (b) Look at the tuning curves: how many peaks does each have, and how far apart are they?

*Your answer:* 

### Exercise 2 · Z-score the neurons

Slide *Units Matter*: distance-based methods, PCA and penalised models all care about the scale of each feature. Some neurons here are thousands of times more variable than others:

In [ ]:
mean = X.mean(axis=0)    # one mean per neuron
std = X.std(axis=0)      # one standard deviation per neuron
plot_std(std)

Z-score every neuron: subtract its mean and divide by its standard deviation, so that each column of `Z` has mean 0 and standard deviation 1. One line, using `X`, `mean` and `std` (numpy broadcasting does the rest).

In [ ]:
Z = ____

**Checkpoint.** `StandardScaler` from scikit-learn does exactly the same thing:

In [ ]:
assert Z.shape == X.shape, "Z should have the same shape as X"
assert np.allclose(Z.mean(axis=0), 0, atol=1e-6), "each neuron (column) should have mean 0"
assert np.allclose(Z.std(axis=0), 1), "each neuron (column) should have standard deviation 1"
assert np.allclose(StandardScaler().fit_transform(X), Z), "StandardScaler should give the same Z"
print("All good: every neuron now has mean 0 and standard deviation 1.")

**Question.** `mean` and `std` were computed on all 2,048 trials. Why would it be a mistake to feed `Z` to a cross-validated decoder?

*Your answer:* 

### Exercise 3 · A decoder in a pipeline, cross-validated

Slide *The Fix: Pipelines*. The decoder chains three steps, and every step that learns something from the data lives inside the pipeline:
1. `StandardScaler()`: your z-score, now learned on the training folds only;
2. `PCA`: compress the 11,302 neurons to **100** components. It is fast, and it leaves 100 × 32 weights to learn instead of 360,000;
3. `LogisticRegression`: Monday's model, with one output per class.

Then cross-validate it with **5** folds. Use the splitter that keeps the 32 directions in the same proportions in every fold (Tuesday: *stratified*). Note that we pass the raw responses `X`: the pipeline does the scaling.

This cell takes a few seconds.

In [ ]:
decoder = make_pipeline(
    StandardScaler(),
    PCA(n_components=____, random_state=SEED),
    LogisticRegression(max_iter=1000),
)
cv = ____(n_splits=____, shuffle=True, random_state=SEED)
scores = cross_val_score(decoder, X, y, cv=cv)

print("accuracy per fold:", np.round(scores, 3))
print(f"mean accuracy: {scores.mean():.1%} ± {scores.std():.1%}")

**Checkpoint.** You should get about 77% (between about 72% and 80% on each fold).

In [ ]:
assert len(scores) == 5, "use 5 folds"
assert 0.70 < scores.mean() < 0.85, f"expected about 77%, got {scores.mean():.1%}: check the pipeline"
print(f"All good: the decoder gets {scores.mean():.1%} of held-out trials exactly right.")

**Question.** Why do the scaler and the PCA go *inside* the pipeline, instead of being run once on all 2,048 trials before cross-validating?

*Your answer:* 

### Exercise 4 · Is 77% good? Compare with a baseline

Slide *Always Compare to a Baseline*. Two references:
- `chance`: the accuracy of guessing at random among the 32 directions;
- a `DummyClassifier` that ignores the neurons and always predicts the **most frequent** class in its training data (that is also the name of its `strategy`, with an underscore). Evaluate it with the same folds `cv`, so the comparison is fair.

In [ ]:
chance = ____
dummy = DummyClassifier(strategy=____)
dummy_scores = cross_val_score(dummy, X, y, cv=cv)

print(f"chance:              {chance:.1%}")
print(f"dummy classifier:    {dummy_scores.mean():.1%}")
print(f"logistic regression: {scores.mean():.1%}  ({scores.mean() / chance:.0f} times chance)")

**Checkpoint.**

In [ ]:
assert abs(chance - 0.03125) < 1e-9, "chance is one in 32"
assert dummy_scores.mean() < 0.05, "the dummy should be close to chance: does it always predict the most frequent class?"
print("All good.")
plot_models({"logistic regression": scores, "dummy": dummy_scores}, chance)

**Question.** The dummy scores 2.9%, a bit below 1/32 = 3.1%. Why? And why run a dummy at all, when chance is so easy to compute?

*Your answer:* 

### Exercise 5 · Which mistakes? The confusion matrix

Slide *The Confusion Matrix*. To build it we need a prediction for every trial, each made by a model that never saw that trial during training. `cross_val_predict` does exactly this: same folds, same pipeline, but it returns the predictions instead of the scores.

Then normalise each row of the confusion matrix, so that row `c` tells you where the trials of direction `c` ended up. The argument `normalize` accepts `"true"`, `"pred"` or `"all"`; the rows are the *true* classes. This cell also takes a few seconds.

In [ ]:
y_pred = ____(decoder, X, y, cv=cv)
conf = confusion_matrix(y, y_pred, normalize=____)
plot_confusion(conf)

**Checkpoint.** You should see a dark diagonal: the correct answers. The colour scale is logarithmic, so the faint cells are rare mistakes, and white cells never happened.

In [ ]:
assert y_pred.shape == y.shape, "y_pred should hold one prediction per trial"
assert np.allclose(conf.sum(axis=1), 1), "each row should sum to 1: normalise over the true classes"
assert abs(np.mean(y_pred == y) - scores.mean()) < 0.01, "the accuracy of y_pred should match Exercise 3"
print(f"All good: {np.mean(y_pred == y):.1%} of trials on the diagonal.")

**Question.** Where do the mistakes fall? Look just next to the diagonal, in the top-right and bottom-left corners, and about 16 steps away from the diagonal.

*Your answer:* 

### Exercise 6 · How far off are the mistakes?

Measure each error as a distance around the circle, in steps of 11.25°: 0 means correct, 1 means a neighbour, 16 means the opposite direction. On a circle of 32 steps, going from class 1 to class 31 is 2 steps, not 30: take the shorter way round. Then compute the accuracy *within ±1 direction*: the fraction of trials with an error of at most one step.

In [ ]:
step = np.abs(y_pred - y)              # 0 ... 31 steps one way round
step = np.minimum(step, ____)   # the shorter way round: 0 ... 16
within1 = np.mean(step ____)

print(f"exact:              {np.mean(step == 0):.1%}")
print(f"within ±1 direction: {within1:.1%}")
print(f"opposite (16 steps): {np.mean(step == 16):.1%}")
plot_steps(step)

**Checkpoint.** On the (logarithmic) histogram you should see a tall bar at 0, a shorter one at 1, almost nothing in the middle, and a small bump again at 15–16.

In [ ]:
assert step.max() <= 16, "the distance around a circle of 32 steps is at most 16"
assert 0.93 < within1 < 0.99, f"expected about 96% within one step, got {within1:.1%}"
print(f"All good: {within1:.0%} of trials are within one direction (11.25°) of the truth.")

**Question.** Exact accuracy is 77%, accuracy within ±1 direction is 96%. Which would you report? What does the gap tell you about the neurons, and about the way we set up the problem?

*Your answer:* 

---
# ★★ Stretch: other models, other folds (about 20 min)

### Stretch 1 · k-NN vs. SVM vs. random forest

Put this morning's models in the same pipeline (scaler, then PCA to 100 components), and evaluate them with the same folds `cv`. You choose the knobs:
- **k-NN**: how many neighbours `k`? Each training fold has about 51 trials per direction.
- **SVM**: which kernel, and which `C`? (The default is `kernel="rbf"`, `C=1`.)
- **random forest**: how many trees? More trees are better but slower; 200 takes a few seconds per fold.

Logistic regression is already in `results`, from Exercise 3. This cell takes about half a minute.

In [ ]:
k = ____
models = {
    "k-NN": KNeighborsClassifier(n_neighbors=k),
    "SVM": SVC(kernel=____, C=____),
    "random forest": RandomForestClassifier(n_estimators=____, random_state=SEED, n_jobs=-1),
}
results = {"logistic regression": scores}
for name, model in models.items():
    pipe = make_pipeline(StandardScaler(), PCA(n_components=100, random_state=SEED), model)
    # YOUR CODE HERE
    # HINT: the same call as in Exercise 3, with pipe instead of decoder
    ...
    print(f"{name:14s} {results[name].mean():.1%} ± {results[name].std():.1%}")

**Checkpoint.** Every model should beat chance by a wide margin. For reference, with k = 15, an RBF kernel with C = 10, and 200 trees, we got: SVM about 83%, logistic regression 77%, random forest about 72%, k-NN about 57%.

In [ ]:
assert all(r.mean() > 0.3 for r in results.values()), "every model should be well above chance"
plot_models(results, chance)

**Question.** Rank the models. Why might k-NN come last on these data, and why do linear boundaries (logistic regression, and the SVM's) do so well?

*Your answer:* 

### Stretch 2 · Shuffled vs. blocked folds

The trials are stored in the order they were shown to the mouse. Over a recording session the animal's state and the recording itself can drift slowly. Tuesday's slide *Data Leakage* warned that shuffling a time series lets the model "train on tomorrow to predict today": with shuffled folds, every test trial has training trials recorded just before and just after it.

Make `blocked`, a 5-fold splitter whose folds are *contiguous* blocks of the session (no shuffling), and use it to get cross-validated predictions `y_pred_blocked` from the same `decoder`. The shuffled predictions are `y_pred`, from Exercise 5.

In [ ]:
# YOUR CODE HERE
# HINT: KFold without shuffling cuts the trials into 5 consecutive blocks
...

correct_shuffled, correct_blocked = (y_pred == y), (y_pred_blocked == y)
print(f"shuffled folds: {correct_shuffled.mean():.1%}")
print(f"blocked folds:  {correct_blocked.mean():.1%}\n")
for i, idx in enumerate(np.array_split(np.arange(len(y)), 5)):
    print(f"fifth {i + 1} of the session: shuffled {correct_shuffled[idx].mean():.0%}   blocked {correct_blocked[idx].mean():.0%}")
plot_session(correct_shuffled, correct_blocked)

**Checkpoint.** You should see two overall accuracies within a percentage point of each other, and two curves that go up and down together along the session.

In [ ]:
assert y_pred_blocked.shape == y.shape, "y_pred_blocked should hold one prediction per trial"
assert not np.array_equal(y_pred_blocked, y_pred), "use the blocked splitter, not cv"
print(f"Difference, shuffled minus blocked: {100 * (correct_shuffled.mean() - correct_blocked.mean()):+.1f} percentage points")

**Question.** Did shuffling inflate the score here? Why not, and why does the accuracy still change so much along the session? Which scheme would you report on another dataset?

*Your answer:* 

---
# ★★★ Challenge: the loop and its dimension

Back to this morning's geometry. In this part we only *look* at the data, nothing is being tested, so we use the z-scored `Z` from Exercise 2.

### Challenge 1 · The loop

Slide *A Loop in Neural Space*. Average the 64 repeats of each direction in `Z`: 32 points in 11,302 dimensions, stacked into `M` (32 × 11,302). Run PCA on `M` and keep three components, `P3` (32 × 3). `plot_loop(P3)` draws them in 3-D, coloured by direction and joined in order.

In [ ]:
# YOUR CODE HERE
# HINT: M[c] is the mean of the rows of Z where y == c; then PCA with 3 components
...
plot_loop(P3)

How much of the variance do the top components capture, for the averages and for single trials (slide *Signal and Noise*)? This cell takes a few seconds.

In [ ]:
pca_M = PCA().fit(M)
pca_trials = PCA(n_components=100, svd_solver="randomized", random_state=SEED).fit(Z)
print(f"PC1 explains {pca_M.explained_variance_ratio_[0]:.0%} of the variance of the 32 averages,")
print(f"but only {pca_trials.explained_variance_ratio_[0]:.1%} of the variance of the single trials.")

**Question.** Follow the colours around the loop: how many times does it wind around? Why? And what do the two variance numbers say about single trials?

*Your answer:* 

### Challenge 2 · Two neighbours are enough: implement 2NN

Slide *Two Neighbours Are Enough* (Facco, d'Errico, Rodriguez & Laio, Sci. Rep. 2017). For every point, take the distances to its first and second nearest neighbours, `r1` and `r2`, and their ratio `mu = r2 / r1`. If the data locally fill a `d`-dimensional space, then

$$-\log\big(1 - F(\mu)\big) = d \, \log \mu ,$$

where $F$ is the cumulative distribution of $\mu$. So:
1. sort the `mu` values; the empirical $F$ of the $i$-th smallest is $i / N$;
2. discard the largest 10% of the `mu` values (they are noisy, and $F = 1$ would give $\log 0$);
3. fit a straight line *through the origin* to the points $(\log\mu, -\log(1-F))$: the least-squares slope is $d = \sum x z / \sum x^2$.

Complete `twonn`, then check it on a Swiss roll, whose answer you know.

In [ ]:
def twonn(Y, discard=0.1, return_curve=False):
    """2NN intrinsic dimension of the points Y (n_points x n_features)."""
    dist, _ = NearestNeighbors(n_neighbors=3).fit(Y).kneighbors(Y)    # dist[:, 0] = 0: the point itself
    # YOUR CODE HERE
    # HINT: the ratio of the second-neighbour distance to the first, sorted
    ...
    n = len(mu)
    F = np.arange(1, n + 1) / n                  # empirical cumulative distribution
    keep_n = int(n * (1 - discard))              # drop the largest 10% of mu
    # YOUR CODE HERE
    # HINT: x = log(mu) and z = -log(1 - F) over the kept points; then the slope through the origin
    ...
    return (d, x, z) if return_curve else d


X_roll, _ = make_swiss_roll(2000, noise=0.0, random_state=1)    # a 2-D sheet rolled up in 3-D
d_roll, x_roll, z_roll = twonn(X_roll, return_curve=True)
plot_twonn(x_roll, z_roll, d_roll)

**Checkpoint.** The Swiss roll is a sheet: you should get a straight line of slope about 2.

In [ ]:
assert 1.8 < d_roll < 2.2, f"expected about 2 for the Swiss roll, got {d_roll:.2f}"
print(f"All good: 2NN dimension of the Swiss roll = {d_roll:.2f}")

### Challenge 3 · Noise and scale

Slide *Noise and Scale*. Measure the 2NN dimension of the 2,048 single trials after keeping only their top `k_pc` principal components, for each `k_pc` in `KPC`, and store the results in the list `id_single`. Use the projections `Zp` from `pca_trials` (Challenge 1).

Then do the same for *averages of 16 repeats*: shuffle the 64 trials of each direction and average them in 4 groups of 16, appending each average to the list `avg16`. That gives 128 points with much less noise. They are projected on their own top 100 components, `Za`; measure their dimension for each `k_pc` up to 50 (128 points cannot support 100 dimensions) and store it in `id_avg16`.

In [ ]:
KPC = [2, 3, 5, 10, 20, 50, 100]
Zp = pca_trials.transform(Z)            # single trials, top 100 PCs
# YOUR CODE HERE
# HINT: one twonn call per k_pc, on the first k_pc columns of Zp
...

rng = np.random.default_rng(45)         # the same draw as on the slide
avg16 = []
for c in range(32):
    # YOUR CODE HERE
    # HINT: rng.permutation(np.where(y == c)[0]) shuffles the trials of direction c; append to avg16 the mean of Z over each group of 16
    ...
Za = PCA(n_components=100, svd_solver="randomized", random_state=SEED).fit_transform(np.array(avg16))
# YOUR CODE HERE
# HINT: as for id_single, on Za and for KPC[:-1]
...

print("PCs kept:     ", KPC)
print("single trials:", np.round(id_single, 1))
print("averages:     ", np.round(id_avg16, 1))
plot_id(KPC, id_single, id_avg16)

**Checkpoint.** You should see the single-trial dimension grow from about 2 (with 2 PCs) to about 16 (with 100 PCs), and the averages stay lower.

In [ ]:
assert len(avg16) == 128, "32 directions x 4 groups of 16 = 128 averages"
assert abs(id_single[0] - 2) < 0.5 and id_single[-1] > 10, "single trials: about 2 with 2 PCs, more than 10 with 100"
assert all(a < s for a, s in zip(id_avg16, id_single)), "the averages should have a lower dimension"
print("All good.")

**Question.** Why does the single-trial dimension keep growing as you keep more components? Why is it lower for the averages? So what is "the" dimension of these data? (To go further: try averages of 4 repeats, or fewer trials, and see what moves.)

*Your answer:* 

### Challenge 4 (optional) · One direction behaves oddly

One of the 32 directions does not behave like the others. Find it, and suggest what might have happened. Places to look: your loop from Challenge 1, the accuracy of the decoder for each direction (`y_pred` from Exercise 5), and the gray-screen trials (`istim == 33`) that we dropped at the very start.

In [ ]:
# YOUR CODE HERE
# HINT: for instance, the accuracy for each direction, and the average z-scored response to each stimulus id, gray screen included
...

In [ ]:
# YOUR CODE HERE
...
plt.show()

In [ ]:
# YOUR CODE HERE
...

**Question.** Which direction is it, what gives it away, and what do you think happened?

*Your answer:* 

---
## Wrap-up

- **What you saw.** A linear decoder reads the grating direction from 11,302 neurons on about 77% of held-out trials, against 3% for chance, and almost all its mistakes are one step away: the neurons encode direction smoothly, and their averages trace a low-dimensional loop inside high-dimensional trial-to-trial variability.
- **Why you can trust the number.** It comes from Tuesday's toolkit: a pipeline, so nothing leaks; stratified and blocked folds; a dummy baseline; and a confusion matrix that shows *which* mistakes are made. The same checklist applies to any dataset you curate.
- **Where it comes back.** This is how a lab uses machine learning. Next, DeepLabCut: tracking animals in video, told through the same recipe card (data, model, loss, optimiser), with the same honest evaluation on frames the network has never seen.